In [5]:
import os

In [6]:
files = []
for dirname, dirnames, filenames in os.walk('..\IMS2013-2024'):
    if 'lemmatized' not in dirname and 'Эксперимент' not in dirname:
        for filename in filenames:
            if 'Abstract' not in filename and 'KW' not in filename and 'Anstract' not in filename and '_rus' in filename:
                with open(os.path.join(dirname, filename), 'r', encoding='utf-8') as f:
                    text = f.read()
                files.append([filename, text])

In [7]:
import pandas as pd

In [8]:
df = pd.DataFrame(files, columns=['name', 'text',])

df

,name,text
0,Adrova_IMS_2013_rus.txt,"Исследование алгоритмов хеширования, используе..."
1,Aksarin_IMS_2013_rus.txt,От театра в интернет к интернет-театру\nК.М. А...
2,Arzumanyan_IMS_2013_rus.txt,Визуализация и восприятие информации в\nгумани...
3,Bershadskaya_IMS_2013_rus.txt,Востребованность услуг электронного правительс...
4,Bikkulov_IMS_2013_rus.txt,Подростковые самоубийства\nв обсуждениях блого...
...,...,...
185,Belkin_IMS_2024_rus.txt,1. Введение По мере стремительного развития...
186,MitrofanovaAdamova_IMS_2024_rus.txt,"1. Введение Проект, представленный в данной ст..."
187,MitrofanovaGolubev_IMS_2024_rus.txt,Под тематическим моделированием традиционно...
188,Sukhan_IMS_2024_rus.txt,1. Введение: типы метаинформации в корпусе и н...


## Sumy

In [5]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Андрей\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
summarizer = LexRankSummarizer()

In [7]:
from tqdm import tqdm
tqdm.pandas()

In [8]:
def summarize(text):
    parser = PlaintextParser.from_string(text, Tokenizer('russian'))
    summary = summarizer(parser.document, 3)
    return '\n'.join([str(sentence) for sentence in summary])

In [9]:
df['abstract'] = df.text.progress_apply(lambda x: summarize(x))

100%|██████████| 190/190 [00:50<00:00,  3.75it/s]


In [10]:
df.to_excel('annotations_sumy.xlsx', index=False)

In [11]:
df.head()

,name,text,abstract
0,Adrova_IMS_2013_rus.txt,"Исследование алгоритмов хеширования, используе...",Анализ времени работы стандартных алгоритмов х...
1,Aksarin_IMS_2013_rus.txt,От театра в интернет к интернет-театру\nК.М. А...,Театр в интернет В истории существования театр...
2,Arzumanyan_IMS_2013_rus.txt,Визуализация и восприятие информации в\nгумани...,Новые возможности не приводят к упрощению науч...
3,Bershadskaya_IMS_2013_rus.txt,Востребованность услуг электронного правительс...,На современном этапе бурного развития социальн...
4,Bikkulov_IMS_2013_rus.txt,Подростковые самоубийства\nв обсуждениях блого...,"Как мы видим, рост обсуждений этой тематики в ..."


# T5

In [1]:
import pandas as pd
df = pd.read_excel('annotations_sumy.xlsx')

In [2]:
from transformers import AutoModelForSeq2SeqLM, T5TokenizerFast

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelWithLMHead, T5ForConditionalGeneration

##RuT5-base-sum

In [4]:
model_name = "IlyaGusev/rut5_base_sum_gazeta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/828k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/977M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/977M [00:00<?, ?B/s]

In [5]:
for i, text in enumerate(df.text[:5]):
  article_text = text
  input_ids = tokenizer([article_text], max_length=600, add_special_tokens=True, padding="max_length", truncation=True, return_tensors="pt")["input_ids"]
  output_ids = model.generate(input_ids=input_ids, no_repeat_ngram_size=4)[0]
  summary = tokenizer.decode(output_ids, skip_special_tokens=True)
  print(f'Аннотация текста {i}:\n {summary}\n\n')

Аннотация текста 0:
 Для решения проблемы хранения пароля пользователя в Web-приложениях необходимо использовать соль, причем уникальную для каждого пользователя, а также использовать радужные таблицы, содержащие миллиарды пар «пароль – результат хеширования». В настоящее время даже длинные пароли не могут считаться безопасными.


Аннотация текста 1:
 Театр в интернет и интернет-театр стали одним из самых популярных в мире видов зрелищного искусства. В настоящее время традиционные театральные деятели обеспокоены тем, что интернет совсем уведет публику из зала.


Аннотация текста 2:
 Визуализация и восприятие информации в гуманитарных науках актуальна и для научного сообщества. Проблема увеличения неопределенности и энтропии актуальна не только для научных сообществ, но и среди исследователей.


Аннотация текста 3:
 Востребованность услуг электронного правительства: анализ дискуссий в социальных сетях, оценка пользователями новых государственных услуг, а также охвате аудитории по изучае